# Bloom Filter Demonstrations - Concepts of Data Science 2025-2026

**Team members:**
- Junior Kaving
- Rexford Holland

## Overview
This notebook demonstrates and tests our Bloom filter implementation.

In [2]:
import os
import sys

# set the path to the parent directory of the current file
sys.path.insert(0, os.path.abspath(os.path.join("..")))

import matplotlib.pyplot as plt
from src.bloom_filter import BloomFilter

## Section 2 - Demonstration
This section demonstrates basic insert and search operations on the Bloom filter.

In [ ]:
bf = BloomFilter(size=1000, num_hashes=3)

# Insert some words
words = ["apple", "banana", "cat", "dog", "elephant"]
for word in words:
    bf.insert(word)

# Search for inserted words
print("Searching for inserted words:")
for word in words:
    print(f"{word}: {bf.search(word)}")

# Search for words never inserted
print("\nSearching for words never inserted:")
not_inserted = ["tiger", "house", "river"]
for word in not_inserted:
    print(f"{word}: {bf.search(word)}")

## Section 3 - Hash Function Uniformity
This section tests whether the hash functions distribute values
evenly across the bit array for two different data types.

In [ ]:
english_words = [
    "apple", "banana", "cat", "dog", "elephant", "fish", "grape",
    "house", "island", "jungle", "kite", "lemon", "mango", "night",
    "ocean", "piano", "queen", "river", "stone", "tiger"
]

dna_sequences = [
    "ATCGGTA", "GCTATCG", "TTAACGT", "CCGATTA", "AAGCTTG",
    "TGCAATC", "GATTACA", "CTAGGCT", "AATCGGA", "TTGCCAA",
    "GCATCGA", "TAGGCTT", "CGATAGC", "AAGTTCC", "GTACCAT",
    "TCAGGTA", "CGATTAC", "AATGGCC", "TTCGAAT", "GCCTAAG"
]

In [ ]:
array_size = 100
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, data, title in zip(
    axes,
    [english_words, dna_sequences],
    ["English Words", "DNA Sequences"]
):
    positions = []
    bf = BloomFilter(size=array_size, num_hashes=1)
    for item in data:
        positions.extend(bf.get_positions(item))

    ax.bar(range(array_size), [positions.count(i) for i in range(array_size)])
    ax.set_title(f"Hash Distribution - {title}")
    ax.set_xlabel("Position in bit array")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("../results/hash_uniformity.png")
plt.show()

## Section 4 - Correctness Tests
Verifying that the Bloom filter behaves correctly for insert and search operations.

In [ ]:
bf = BloomFilter(size=1000, num_hashes=3)

# inserted words should always be found
words = ["apple", "banana", "cat", "dog", "elephant"]
for word in words:
    bf.insert(word)

for word in words:
    assert bf.search(word) is True, f"{word} should be found"

# empty filter should return False
bf_empty = BloomFilter(size=1000, num_hashes=3)
assert bf_empty.search("apple") is False, "empty filter should return False"

print("All correctness tests passed.")

## Section 5 - Testing With Two Data Types
Verifying that the Bloom filter works correctly for
English words and DNA sequences.

In [ ]:
# English words
bf_english = BloomFilter(size=1000, num_hashes=3)
for word in english_words:
    bf_english.insert(word)

for word in english_words:
    assert bf_english.search(word) is True, f"{word} should be found"

print("All English word tests passed.")

# DNA sequences
bf_dna = BloomFilter(size=1000, num_hashes=3)
for seq in dna_sequences:
    bf_dna.insert(seq)

for seq in dna_sequences:
    assert bf_dna.search(seq) is True, f"{seq} should be found"

print("All DNA sequence tests passed.")

## Section 6 - False Positive Rate
Measuring how the false positive rate grows as more words are inserted,
including beyond the filter's designed capacity.

In [ ]:
import random
import string

def measure_false_positive_rate(
    bf: BloomFilter,
    test_words: list[str]
) -> float:
    false_positives = sum(
        1 for word in test_words if bf.search(word)
    )
    return false_positives / len(test_words)

In [ ]:
# Generate a large set of words to insert and test with
all_words = [f"word{i}" for i in range(5000)]
test_words = [f"test{i}" for i in range(1000)]

# Filter designed for 1000 words
steps = [100, 200, 300, 500, 700, 1000, 1500, 2000, 3000, 5000]
false_positive_rates = []

for n in steps:
    bf = BloomFilter(size=10000, num_hashes=3)
    for word in all_words[:n]:
        bf.insert(word)
    rate = measure_false_positive_rate(bf, test_words)
    false_positive_rates.append(rate)
    print(f"Inserted {n} words -> false positive rate: {rate:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(steps, false_positive_rates, marker="o")
plt.axvline(x=1000, color="red", linestyle="--", label="Design capacity")
plt.xlabel("Number of words inserted")
plt.ylabel("False positive rate")
plt.title("False Positive Rate vs Words Inserted")
plt.legend()
plt.savefig("../results/false_positive_rate.png")
plt.show()

## Section 7 - Compression Rate
Measuring the memory efficiency of the Bloom filter compared
to storing the actual words in a Python set.

In [ ]:
import sys

def measure_compression_rate(
    num_words: int,
    array_size: int,
    num_hashes: int
) -> float:
    words = [f"word{i}" for i in range(num_words)]
    
    # memory used by actual set
    actual_set = set(words)
    actual_memory = sys.getsizeof(actual_set)
    
    # memory used by bloom filter bit array
    bf = BloomFilter(size=array_size, num_hashes=num_hashes)
    bloom_memory = sys.getsizeof(bf.bit_array)
    
    return actual_memory / bloom_memory

In [ ]:
word_counts = [100, 500, 1000, 2000, 5000, 10000]
compression_rates = [
    measure_compression_rate(n, array_size=10000, num_hashes=3)
    for n in word_counts
]

plt.figure(figsize=(10, 5))
plt.plot(word_counts, compression_rates, marker="o")
plt.xlabel("Number of words")
plt.ylabel("Compression rate")
plt.title("Compression Rate vs Number of Words")
plt.savefig("../results/compression_vs_words.png")
plt.show()

false_positive_targets = [0.01, 0.02, 0.05, 0.10, 0.20]
array_sizes = [95851, 47926, 23963, 14378, 9585]  # computed for 1000 words

compression_rates_fp = [
    measure_compression_rate(1000, array_size=size, num_hashes=3)
    for size in array_sizes
]

plt.figure(figsize=(10, 5))
plt.plot(false_positive_targets, compression_rates_fp, marker="o")
plt.xlabel("Target false positive rate")
plt.ylabel("Compression rate")
plt.title("Compression Rate vs False Positive Rate")
plt.savefig("../results/compression_vs_fp_rate.png")
plt.show()

# Section 8 - Complexity Analysis

### Complexity Overview
* **Time Complexity**: O(k) for both insert and search, where k is the number of hash functions.
* **Space Complexity**: O(m) using a fixed bit array of size m, without storing the actual items.

### Empirical Benchmarking
The script below measures the average execution time per operation across different dataset sizes to verify the theoretical O(k) time complexity.

In [ ]:
import time
import matplotlib.pyplot as plt

sizes = [100, 500, 1000, 5000, 10000, 50000]
insert_times = []
search_times = []

for n in sizes:
    words = [f"word{i}" for i in range(n)]
    bf = BloomFilter(size=n * 10, num_hashes=3)

    start = time.perf_counter()
    for w in words:
        bf.insert(w)
    insert_times.append((time.perf_counter() - start) / n)

    start = time.perf_counter()
    for w in words:
        bf.search(w)
    search_times.append((time.perf_counter() - start) / n)

plt.figure(figsize=(10, 5))
plt.plot(sizes, insert_times, marker="o", label="insert")
plt.plot(sizes, search_times, marker="s", label="search")
plt.xlabel("Number of items")
plt.ylabel("Average time per operation (s)")
plt.title("Time Complexity - insert and search scale as O(k)")
plt.legend()
plt.savefig("../results/complexity.png")
plt.show()

print("insert and search both run in O(k) time regardless of n — confirmed by flat lines above")